# Feature Engineering: rolling-статистики матчей

Для каждого матча считаем агрегированные статистики прошлых матчей на двух уровнях:
- **Team-level** (`generate_rolling_features`) - по паре игроков как единому объекту
- **Player-level** (`generate_player_rolling_features`) - индивидуально по каждому игроку, затем для команды две оценки игроков усредняются

Параметры:
- `stat_columns` - базовые имена колонок (без суффикса `_team_1`/`_team_2`)
- `windows` - размеры окна (кол-во прошлых матчей для агрегации). `None` = вся история
- `lags` - отступы (0 = последний прошлый матч включён, 1 = пропустить и т.д.)
- `agg_funcs` - функции агрегации (`{имя: pandas_agg}`)
- `perspectives` - `own` (агрегация по своей статистике), `opp` (агрегация по статистике соперников)
- `slice_by` (только для team-level),  если задан столбец (например, `tournament_level`), rolling считается отдельно для каждого значения среза

Результат сохраняется в `data/features/rolling_stats.csv`

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("../../data/processed")
FEATURES = Path("../../data/features")
FEATURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED / "matches.csv", parse_dates=["played_at"])
df = df.sort_values("played_at").reset_index(drop=True)
print(f"Матчей: {len(df)}")

Матчей: 2549


## Team-level rolling

In [15]:
def _shorten(stat_name):
    for prefix in ["stats_match_", "stats_set_1_", "stats_set_2_", "stats_set_3_"]:
        if stat_name.startswith(prefix):
            return stat_name[len(prefix):]
    return stat_name


def _compute_rolling_on_series(series, windows, lags, agg_funcs):
    result = {}
    for lag in lags:
        shifted = series.shift(lag + 1)  # +1: текущий матч НИКОГДА не включается
        for w in windows:
            if w is None:
                rolled = shifted.expanding(min_periods=1)
                w_str = "all"
            else:
                rolled = shifted.rolling(w, min_periods=1)
                w_str = str(w)
            for agg_name, agg_func in agg_funcs.items():
                key = f"{agg_name}_{w_str}w_{lag}lag"
                result[key] = rolled.agg(agg_func)
    return result


def generate_rolling_features(
    df,
    stat_columns,
    windows=(5, 10),
    lags=(0,),
    agg_funcs=None,
    perspectives=("own",),
    slice_by=None,
):
    if agg_funcs is None:
        agg_funcs = {"mean": "mean"}

    df_sorted = df.sort_values("played_at").reset_index(drop=True)

    # Team keys
    team1_key = list(zip(
        np.minimum(df_sorted["player_id_1"], df_sorted["player_id_2"]),
        np.maximum(df_sorted["player_id_1"], df_sorted["player_id_2"]),
    ))
    team2_key = list(zip(
        np.minimum(df_sorted["player_id_3"], df_sorted["player_id_4"]),
        np.maximum(df_sorted["player_id_3"], df_sorted["player_id_4"]),
    ))

    keep = ["match_id", "played_at"]
    if slice_by:
        keep.append(slice_by)

    t1 = df_sorted[keep].copy()
    t1["team"] = team1_key
    t1["_side"] = "team_1"
    for stat in stat_columns:
        t1[f"own_{stat}"] = df_sorted[f"{stat}_team_1"].values
        t1[f"opp_{stat}"] = df_sorted[f"{stat}_team_2"].values

    t2 = df_sorted[keep].copy()
    t2["team"] = team2_key
    t2["_side"] = "team_2"
    for stat in stat_columns:
        t2[f"own_{stat}"] = df_sorted[f"{stat}_team_2"].values
        t2[f"opp_{stat}"] = df_sorted[f"{stat}_team_1"].values

    hist = pd.concat([t1, t2], ignore_index=True).sort_values("played_at")

    if slice_by:
        slice_values = sorted(df_sorted[slice_by].dropna().unique())
    else:
        slice_values = [None]

    feat_parts = []

    for sv in slice_values:
        sub = hist[hist[slice_by] == sv] if sv is not None else hist
        suffix_slice = f"_{slice_by}_{sv}" if sv is not None else ""

        grp_blocks = []
        for _, grp in sub.groupby("team"):
            grp = grp.sort_values("played_at")
            block = {}
            for persp in perspectives:
                for stat in stat_columns:
                    short = _shorten(stat)
                    rolls = _compute_rolling_on_series(
                        grp[f"{persp}_{stat}"], windows, lags, agg_funcs
                    )
                    for key, vals in rolls.items():
                        col = f"roll_{persp}_{short}_{key}{suffix_slice}"
                        block[col] = vals
            grp_blocks.append(pd.DataFrame(block, index=grp.index))

        feat_block = (
            pd.concat(grp_blocks, axis=0).reindex(sub.index)
            if grp_blocks
            else pd.DataFrame(index=sub.index)
        )
        feat_block["match_id"] = sub["match_id"].values
        feat_block["_side"] = sub["_side"].values
        feat_parts.append(feat_block)

    merged = pd.concat(feat_parts, axis=1)
    merged = merged.loc[:, ~merged.columns.duplicated()]

    feat_cols = [c for c in merged.columns if c.startswith("roll_")]

    t1_f = merged[merged["_side"] == "team_1"][["match_id"] + feat_cols].copy()
    t1_f = t1_f.rename(columns={c: f"team1_{c}" for c in feat_cols})

    t2_f = merged[merged["_side"] == "team_2"][["match_id"] + feat_cols].copy()
    t2_f = t2_f.rename(columns={c: f"team2_{c}" for c in feat_cols})

    return t1_f.merge(t2_f, on="match_id", how="outer")

## Player-level rolling

In [16]:
def generate_player_rolling_features(
    df,
    stat_columns,
    windows=(5, 10),
    lags=(0,),
    agg_funcs=None,
    perspectives=("own",),
):
    """
    Rolling статистики на уровне каждого игрока
    Для каждой команды: среднее rolling двух игроков
    """
    if agg_funcs is None:
        agg_funcs = {"mean": "mean"}

    df_sorted = df.sort_values("played_at").reset_index(drop=True)

    # 4 записи на матч: по одной на каждого игрока
    player_records = []
    for pid_col, side in [
        ("player_id_1", "team_1"), ("player_id_2", "team_1"),
        ("player_id_3", "team_2"), ("player_id_4", "team_2"),
    ]:
        rec = df_sorted[["match_id", "played_at"]].copy()
        rec["player_id"] = df_sorted[pid_col].values
        rec["_side"] = side
        rec["_pid_col"] = pid_col
        for stat in stat_columns:
            rec[f"own_{stat}"] = df_sorted[f"{stat}_{side}"].values
            opp_side = "team_2" if side == "team_1" else "team_1"
            rec[f"opp_{stat}"] = df_sorted[f"{stat}_{opp_side}"].values
        player_records.append(rec)

    phist = pd.concat(player_records, ignore_index=True).sort_values("played_at")

    grp_blocks = []
    for _, grp in phist.groupby("player_id"):
        grp = grp.sort_values("played_at")
        block = {}
        for persp in perspectives:
            for stat in stat_columns:
                short = _shorten(stat)
                rolls = _compute_rolling_on_series(
                    grp[f"{persp}_{stat}"], windows, lags, agg_funcs
                )
                for key, vals in rolls.items():
                    col = f"proll_{persp}_{short}_{key}"
                    block[col] = vals
        grp_blocks.append(pd.DataFrame(block, index=grp.index))

    feat_block = (
        pd.concat(grp_blocks, axis=0).reindex(phist.index)
        if grp_blocks
        else pd.DataFrame(index=phist.index)
    )
    feat_block["match_id"] = phist["match_id"].values
    feat_block["_side"] = phist["_side"].values
    feat_block["_pid_col"] = phist["_pid_col"].values

    feat_cols = [c for c in feat_block.columns if c.startswith("proll_")]

    # Для каждой стороны усредняем двух игроков
    parts = []
    for side, prefix in [("team_1", "team1"), ("team_2", "team2")]:
        side_data = feat_block[feat_block["_side"] == side][["match_id"] + feat_cols]
        averaged = side_data.groupby("match_id")[feat_cols].mean()
        averaged = averaged.rename(columns={c: f"{prefix}_{c}" for c in feat_cols})
        parts.append(averaged)

    result = parts[0].join(parts[1], how="outer").reset_index()
    return result

## Генерация фич

In [17]:
MATCH_STATS = [
    "stats_match_total_points_won",
    "stats_match_break_points_converted",
    "stats_match_longest_streak",
    "stats_match_aces",
    "stats_match_double_faults",
    "stats_match_won_on_1st_serve",
    "stats_match_won_on_2nd_serve",
    "stats_match_service_games",
    "stats_match_won_on_1st_return",
    "stats_match_won_on_2nd_return",
    "stats_match_return_games",
    "stats_match_total_won_on_serve",
    "stats_match_total_won_on_return",
]

In [18]:
rolling_team = generate_rolling_features(
    df,
    stat_columns=MATCH_STATS,
    windows=[5, 10, None],
    lags=[0],
    agg_funcs={"mean": "mean", "median": "median"},
    perspectives=["own", "opp"],
)
print(f"Team-level rolling: {rolling_team.shape[1] - 1} фич")

Team-level rolling: 312 фич


In [19]:
rolling_player = generate_player_rolling_features(
    df,
    stat_columns=MATCH_STATS,
    windows=[5, 10, None],
    lags=[0],
    agg_funcs={"mean": "mean"},
    perspectives=["own"],
)
print(f"Player-level rolling: {rolling_player.shape[1] - 1} фич")

Player-level rolling: 78 фич


In [20]:
rolling_by_level = generate_rolling_features(
    df,
    stat_columns=MATCH_STATS,
    windows=[5],
    lags=[0],
    agg_funcs={"mean": "mean"},
    perspectives=["own"],
    slice_by="tournament_level",
)
print(f"Rolling по уровню турнира: {rolling_by_level.shape[1] - 1} фич")

Rolling по уровню турнира: 130 фич


In [21]:
features = rolling_team.merge(rolling_player, on="match_id", how="outer")
features = features.merge(rolling_by_level, on="match_id", how="outer")

print(f"Итого: {features.shape[1] - 1} фич, {features.shape[0]} матчей")
print()
print("Пример столбцов:")
for c in list(features.columns[:6]):
    print(f"  {c}")
print("  ...")
player_cols = [c for c in features.columns if "proll" in c]
for c in player_cols[:4]:
    print(f"  {c}")

Итого: 520 фич, 2549 матчей

Пример столбцов:
  match_id
  team1_roll_own_total_points_won_mean_5w_0lag
  team1_roll_own_total_points_won_median_5w_0lag
  team1_roll_own_total_points_won_mean_10w_0lag
  team1_roll_own_total_points_won_median_10w_0lag
  team1_roll_own_total_points_won_mean_allw_0lag
  ...
  team1_proll_own_total_points_won_mean_5w_0lag
  team1_proll_own_total_points_won_mean_10w_0lag
  team1_proll_own_total_points_won_mean_allw_0lag
  team1_proll_own_break_points_converted_mean_5w_0lag


In [28]:
out_path = FEATURES / "rolling_stats.csv"
features.to_csv(out_path, index=False)